# Zen - model server on Kaggle: Qwen 3.6 35B-A3B and EmbeddingGemma 2

Runs **llama-server** (llama.cpp's own Linux CUDA build) with **Qwen 3.6 35B-A3B** on Kaggle's two T4 GPUs, with
Qwen's own settings, and **EmbeddingGemma 2** - Zen's search by meaning, pictures too - on the second GPU. It opens
one public address - protected by a password - for Zen on your PC
(Zen-harness.exe asks for the two, or put them in `kaggle/bridge.env` for the bridge).

**Before running** (right-hand panel, *Session options*):
1. **Accelerator:** `GPU T4 x2` (the model is split over both automatically).
2. **Internet:** `On` (Kaggle asks for a phone-verified account).
3. **Persistence:** `No persistence` - a kept llama-server comes back broken, so it is downloaded fresh.
4. *Optional:* **Add-ons → Secrets** → add `LLAMA_API_KEY` with a password of your choice, so it stays the same every session.

Then **Run All**. It downloads llama-server (two builds, about a minute each) and the models (about 22 GB: a few
minutes).
The last cell prints the **address** and **password** for Zen and keeps the server running -
keep this tab open (or use *Save Version → Save & Run All* to run it in the background for up to 12 hours).
Stop the last cell, or the session, to stop serving.

What it runs (measured 2026-10-17): 2 slots of 64k tokens each (one for the chat, one for Zen's helpers), about
58 words/s on an empty slot and 33 when a slot is nearly full; about 7 GB of the GPUs stay free, about 4 GB of
them on the second - where the search model goes. Zen finds the search model by itself (Zen-harness.exe; from
source, two settings: see Zen's README); an older notebook without it still works, with Zen's small model.

In [ ]:
# --- Settings -------------------------------------------------------------------------------------
# Qwen 3.6 35B-A3B: 35B in all, about 3B used per word - fast on the T4s. Unsloth's K-quant (UD-Q4_K_S) is the fast
# one here (with Qwen's built-in MTP and the IQ4_XS file it ran 10.6 words/s; this file without MTP, 48+).
REPO = "unsloth/Qwen3.6-35B-A3B-GGUF"
MODEL_FILE = "Qwen3.6-35B-A3B-UD-Q4_K_S.gguf"
VISION_FILE = "mmproj-F16.gguf"   # lets the model read pictures
# Qwen 3.6's own sampling, from its model card: thinking mode, general tasks. The presence penalty keeps long
# thinking from repeating itself. Zen's analyst asks per request for Qwen's precise-coding set (temperature 0.6,
# no penalty: app/assistant/workers/contracts.py).
SAMPLING = ["--temp", "1.0", "--top-p", "0.95", "--top-k", "20", "--min-p", "0.0", "--presence-penalty", "1.5"]
# Two slots - one for the chat, one for Zen's helpers - of 64k tokens each: CONTEXT is both together
CONTEXT = 131072
LLAMA_BUILD = "b11474"   # llama.cpp's ready-made Linux CUDA 12.8 build (Kaggle has CUDA 12.8)
PORT = 8080
# Search by meaning - Zen's files, chats and pictures, its web results, analyze_text: EmbeddingGemma 2 and its picture
# projector on the second GPU, with a llama-server of its own: a newer build knows it (Qwen stays on the build its
# speed was measured on). The same model file as Zen's on a PC, so a project's index is the same either way.
EMBED_REPO = "ggml-org/embeddinggemma-2-GGUF"
EMBED_FILE = "embeddinggemma-2-Q8_0.gguf"
EMBED_VISION = "mmproj-embeddinggemma-2-Q8_0.gguf"
EMBED_BUILD = "b11515"
EMBED_PORT = 8070
EMBED_CONTEXT = 8192   # its whole window: an embedding needs its input in one batch (as app/rag/embed_server.py)
ROUTER_PORT = 8090     # the tunnel's: /embed/... to the search model, everything else to Qwen
WORK = "/kaggle/working"
MODELS = "/tmp/models/qwen36"
EMBED_MODELS = "/tmp/models/embeddinggemma"

import glob, json, os, re, secrets, shutil, subprocess, threading, time, urllib.error, urllib.request

# Runs a command with its output shown here as it comes; on an error, stops with the last lines of
# output so the reason is visible.
def sh(cmd, quiet_progress=False):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tail, steps = [], 0
    for line in proc.stdout:
        tail = (tail + [line])[-40:]
        if quiet_progress and "%]" in line:
            steps += 1
            if steps % 25:
                continue
        print(line, end="", flush=True)
    if proc.wait():
        raise RuntimeError(f"{cmd[0]} failed:\n" + "".join(tail))

# Until a llama-server answers its health check (it loads its model first); stops with its log's end if it quits.
def wait_up(proc, port, log_path, tries=300):
    for _ in range(tries):
        if proc.poll() is not None:
            raise RuntimeError("llama-server stopped:\n" + open(log_path).read()[-3000:])
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=2) as response:
                if json.load(response).get("status") == "ok":
                    return
        except Exception:
            pass
        time.sleep(2)
    raise RuntimeError(f"llama-server on port {port} didn't come up:\n" + open(log_path).read()[-3000:])

def gpus():
    return subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip().replace("\n", " / ")

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)

In [ ]:
# --- llama-server: llama.cpp's ready-made Linux CUDA 12.8 builds (about a minute each), fresh each session:
# a copy kept by Persistence comes back without its run permission and its library links
def llama_build(build):
    """That build's llama-server, and the environment it runs in (its libraries' folders)."""
    bin_dir = f"/tmp/llama-bin/{build}"
    if not glob.glob(f"{bin_dir}/**/llama-server", recursive=True):
        os.makedirs(bin_dir, exist_ok=True)
        base = f"https://github.com/ggml-org/llama.cpp/releases/download/{build}"
        for name in (f"llama-{build}-bin-ubuntu-cuda-12.8-x64.tar.gz",          # llama-server and its libraries
                     f"cudart-llama-{build}-bin-ubuntu-cuda-12.8-x64.tar.gz"):   # the CUDA runtime it was built with
            urllib.request.urlretrieve(f"{base}/{name}", f"/tmp/{name}")
            sh(["tar", "-xzf", f"/tmp/{name}", "-C", bin_dir])
    server = glob.glob(f"{bin_dir}/**/llama-server", recursive=True)[0]
    # where its libraries are: every folder of the download holding one
    env = {**os.environ, "LD_LIBRARY_PATH": ":".join(sorted({os.path.dirname(p) for p in glob.glob(f"{bin_dir}/**/*.so*", recursive=True)})
                                                     + [os.environ.get("LD_LIBRARY_PATH", "")])}
    version = subprocess.run([server, "--version"], capture_output=True, text=True, env=env)
    said = version.stdout + version.stderr
    print(server, "|", next((line for line in said.splitlines() if "version" in line), said.strip()[-300:]))
    return server, env

SERVER, ENV = llama_build(LLAMA_BUILD)              # Qwen's
EMBED_SERVER, EMBED_ENV = llama_build(EMBED_BUILD)  # the search model's

In [ ]:
# --- The models and their vision parts, from Hugging Face (about 21 GB, and 0.9 GB for search) ---------
from huggingface_hub import hf_hub_download

paths = {name: hf_hub_download(REPO, name, local_dir=MODELS) for name in (MODEL_FILE, VISION_FILE)}
paths.update({name: hf_hub_download(EMBED_REPO, name, local_dir=EMBED_MODELS) for name in (EMBED_FILE, EMBED_VISION)})
for name, path in paths.items():
    print(f"{os.path.getsize(path) / 1e9:5.2f} GB  {name}")

In [ ]:
# --- The password: your Kaggle secret LLAMA_API_KEY if set, else this session's (kept when re-run) ----
if "API_KEY" not in globals():
    try:
        from kaggle_secrets import UserSecretsClient
        API_KEY = UserSecretsClient().get_secret("LLAMA_API_KEY")
    except Exception:
        API_KEY = secrets.token_urlsafe(24)

# a server from an earlier run of this cell still holds the port and the GPUs: stop it first
if "server" in globals() and server.poll() is None:
    server.terminate()
    server.wait(60)

# --- Start llama-server: Qwen's sampling, both GPUs, 8-bit conversation memory, 2 slots ---------------
args = [SERVER, "-m", paths[MODEL_FILE], "--alias", "local", *SAMPLING,
        "--mmproj", paths[VISION_FILE], "--image-min-tokens", "280", "--image-max-tokens", "560",
        "--fit", "on", "--fit-target", "256", "--fit-ctx", str(CONTEXT),
        "-c", str(CONTEXT), "-np", "2", "--no-kv-unified", "-fa", "on", "-ctk", "q8_0", "-ctv", "q8_0",
        "-b", "1024", "-ub", "512", "--reasoning", "auto", "--reasoning-format", "deepseek", "--slots",
        "--api-key", API_KEY, "--host", "127.0.0.1", "--port", str(PORT)]
log_path = f"{WORK}/llama-server.log"
server = subprocess.Popen(args, stdout=open(log_path, "w"), stderr=subprocess.STDOUT, env=ENV)
wait_up(server, PORT, log_path)
print("llama-server is up | Qwen 3.6 35B-A3B | two GPUs, 8-bit memory |", gpus())

In [ ]:
# --- Start the search model's llama-server on the second GPU (Qwen leaves about 4 GB free there) ---------
# As Zen runs it on a PC (app/rag/embed_server.py), with the same password as Qwen's.
if "embed" in globals() and embed.poll() is None:
    embed.terminate()
    embed.wait(60)
embed_args = [EMBED_SERVER, "-m", paths[EMBED_FILE], "--mmproj", paths[EMBED_VISION], "--embeddings",
              "-c", str(EMBED_CONTEXT), "-b", str(EMBED_CONTEXT), "-ub", str(EMBED_CONTEXT), "--kv-unified",
              "--no-webui", "--fit", "on", "--api-key", API_KEY, "--host", "127.0.0.1", "--port", str(EMBED_PORT)]
embed_log = f"{WORK}/embed-server.log"
embed = subprocess.Popen(embed_args, stdout=open(embed_log, "w"), stderr=subprocess.STDOUT,
                         env={**EMBED_ENV, "CUDA_VISIBLE_DEVICES": "1"})
wait_up(embed, EMBED_PORT, embed_log)
print("search model is up | EmbeddingGemma 2, pictures too | second GPU |", gpus())

In [ ]:
# --- A quick speed check, straight to the server -----------------------------------------------------
def ask(prompt, max_tokens=300):
    body = {"messages": [{"role": "user", "content": prompt}], "max_tokens": max_tokens,
            "chat_template_kwargs": {"enable_thinking": False}}
    request = urllib.request.Request(f"http://127.0.0.1:{PORT}/v1/chat/completions", json.dumps(body).encode(),
                                     {"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"})
    with urllib.request.urlopen(request, timeout=600) as response:
        out = json.load(response)
    t = out["timings"]
    print(f"prompt {t['prompt_n']} tokens at {t['prompt_per_second']:.0f}/s | "
          f"answer {t['predicted_n']} tokens at {t['predicted_per_second']:.1f}/s")
    print(out["choices"][0]["message"]["content"][:400])

ask("In three sentences: what makes a sales forecast trustworthy?")

In [ ]:
# --- Where the model sits: memory per GPU, and RAM (the model must not spill into RAM) ---------------
print(subprocess.run("nvidia-smi --query-gpu=index,memory.used,utilization.gpu --format=csv; free -g",
                     shell=True, capture_output=True, text=True).stdout)

In [ ]:
# --- One address for both: /embed/... to the search model, everything else to Qwen ----------------------
# (a quick tunnel serves one port). Requests and answers pass as they are - the password too, which both servers
# check - and Qwen's answers stream through as they come; Zen's Stop closes the request, which stops the model.
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

HOP = {"connection", "keep-alive", "transfer-encoding", "content-length", "content-encoding", "host"}

class Router(BaseHTTPRequestHandler):
    routes = {"/embed": EMBED_PORT}  # path prefix -> the port it goes to, without the prefix
    default = PORT

    def log_message(self, *args):
        pass

    def do_GET(self):
        self.forward()

    def do_POST(self):
        self.forward()

    def do_OPTIONS(self):
        self.forward()

    def forward(self):
        port, path = self.default, self.path
        for prefix, to in self.routes.items():
            if path == prefix or path.startswith((prefix + "/", prefix + "?")):
                port, path = to, path[len(prefix):] or "/"
                if path.startswith("?"):
                    path = "/" + path
                break
        body = self.rfile.read(int(self.headers.get("Content-Length") or 0)) or None
        headers = {k: v for k, v in self.headers.items() if k.lower() not in HOP}
        request = urllib.request.Request(f"http://127.0.0.1:{port}{path}", data=body, headers=headers, method=self.command)
        try:
            response = urllib.request.urlopen(request, timeout=600)
        except urllib.error.HTTPError as exc:
            response = exc  # the server's own error (its status and message) goes back as it is
        except (urllib.error.URLError, OSError) as exc:
            data = json.dumps({"error": {"message": f"Nothing answers on the notebook for {self.path} ({exc})"}}).encode()
            self.send_response(502)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)
            return
        with response:
            try:
                self.send_response(getattr(response, "status", None) or response.code)
                for name, value in response.headers.items():
                    if name.lower() not in HOP:
                        self.send_header(name, value)
                if "text/event-stream" in (response.headers.get("Content-Type") or ""):
                    self.send_header("Connection", "close")
                    self.end_headers()
                    for line in response:
                        self.wfile.write(line)
                        self.wfile.flush()
                else:
                    data = response.read()
                    self.send_header("Content-Length", str(len(data)))
                    self.end_headers()
                    self.wfile.write(data)
            except (BrokenPipeError, ConnectionResetError, ConnectionAbortedError):
                pass  # the caller stopped listening (Zen's Stop): leaving this closes the server's side too

# --- start it ---
if "router" in globals():
    router.shutdown()
    router.server_close()
router = ThreadingHTTPServer(("127.0.0.1", ROUTER_PORT), Router)
threading.Thread(target=router.serve_forever, daemon=True, name="router").start()
# both, through it, as Zen will ask: Qwen's health, and a sentence embedded by the search model
with urllib.request.urlopen(f"http://127.0.0.1:{ROUTER_PORT}/health", timeout=10) as response:
    qwen = json.load(response).get("status")
request = urllib.request.Request(f"http://127.0.0.1:{ROUTER_PORT}/embed/embedding",
                                 json.dumps({"content": ["task: search result | query: sales by month"]}).encode(),
                                 {"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"})
with urllib.request.urlopen(request, timeout=60) as response:
    vector = json.load(response)[0]["embedding"]   # one per input; llama-server nests a pooled one in a list
vector = vector[0] if vector and isinstance(vector[0], list) else vector
print(f"router on port {ROUTER_PORT} | Qwen: {qwen} | search: a vector of {len(vector)} numbers")

In [ ]:
# --- The public address: a Cloudflare quick tunnel (no account needed) -------------------------------
CLOUDFLARED = "/tmp/cloudflared"
if not os.path.exists(CLOUDFLARED):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                               CLOUDFLARED)
    os.chmod(CLOUDFLARED, 0o755)
tunnel = subprocess.Popen([CLOUDFLARED, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{ROUTER_PORT}"],
                          stdout=open("/tmp/cloudflared.log", "w"), stderr=subprocess.STDOUT)
URL = None
for _ in range(90):
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/tmp/cloudflared.log").read())
    if found:
        URL = found.group(0)
        break
    time.sleep(1)
if URL is None:
    raise RuntimeError("No tunnel address:\n" + open("/tmp/cloudflared.log").read()[-2000:])
print("=" * 72)
print("For the bridge on your PC:")
print("  address :", URL)
print("  password:", API_KEY)
print("=" * 72)

In [ ]:
# --- Keep serving (stop this cell to stop) ------------------------------------------------------------
# Shows every few minutes that the server and the tunnel are still up; the address stays the same while
# this session runs.
def health(port):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=5) as response:
            return json.load(response).get("status")
    except Exception as exc:
        return f"DOWN ({exc})"

while True:
    print(time.strftime("%H:%M:%S"), "| Qwen:", health(PORT), "| search:", health(EMBED_PORT), "| tunnel:",
          "up" if tunnel.poll() is None else "STOPPED", "|", URL, flush=True)
    time.sleep(300)